# Pipelines de HuggingFace

En esta sesión, nos dirigiremos a Google Colab y usaremos este Notebook para explorar los pipelines de la API de alto nivel de HuggingFace que son geniales.

https://colab.research.google.com/drive/15hH3lhQZtMd1k9N2hYTi3_PmNan0MDk5?usp=share_link

Puedes usar un entorno de ejecución de GPU T4 de bajo costo (o gratuito) para este notebook, ¡y los resultados se ven geniales!

Hay instrucciones en el notebook para configurar tu token de HuggingFace e incluirlo como secreto en el notebook.

# ¡Bienvenido a Pipelines!

La biblioteca de transformadores HuggingFace proporciona API en dos niveles diferentes.

La API de alto nivel para usar modelos de código abierto para tareas de inferencia típicas se llama "pipelines". Es increíblemente fácil de usar.

Se crea una tubería usando algo como:

`my_pipeline = pipeline("tarea_que_quiero_hacer")`

Seguido por

`result = my_pipeline(mi_resultado)`

¡Y eso es todo!

Consulta el final de este laboratorio colaborativo para obtener una lista de todas las tuberías.

## Nota al margen:

Puede que ya sepas esto, pero por si no estás familiarizado con la palabra "inferencia" que utilizo aquí:

Al trabajar con modelos de ciencia de datos, puedes estar llevando a cabo 2 actividades muy diferentes: **entrenamiento** e **inferencia**.

### 1. Entrenamiento

**Entrenamiento** es cuando le proporcionas a un modelo datos para que se adapte y mejore en una tarea en el futuro. Esto se hace actualizando sus configuraciones internas: los parámetros o pesos del modelo. Si estás entrenando un modelo que ya ha tenido algo de entrenamiento, la actividad se llama "ajuste fino".

### 2. Inferencia

**Inferencia** es cuando estás trabajando con un modelo que ya ha sido entrenado_. Estás usando ese modelo para producir nuevos resultados a partir de nuevas entradas, aprovechando todo lo que aprendió mientras se entrenaba. A la inferencia también se la conoce a veces como "Ejecución" o "Ejecución de un modelo".

Todo el uso que hemos hecho de las API para GPT, Claude y Gemini en las últimas semanas son ejemplos de **inferencia**. La "P" de GPT significa "preentrenado", lo que significa que ya se ha entrenado con datos (¡muchos!). En la semana 6, intentaremos ajustar GPT nosotros mismos.

La API de pipelines en HuggingFace solo se utiliza para **inferencia**: ejecutar un modelo que ya se ha entrenado. En la semana 7, entrenaremos nuestro propio modelo y necesitaremos utilizar las API más avanzadas de HuggingFace que veremos en la próxima conferencia.

In [ ]:
!pip install -q transformers datasets diffusers

In [ ]:
# Imports

import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import pipeline
# El pipeline "text-to-speech" está roto en esta versión de transformers, así que la celda de audio
# usa SpeechT5 directamente (ver el comentario de esa celda).
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
from diffusers import DiffusionPipeline
from datasets import load_dataset
import soundfile as sf
from IPython.display import Audio


# Nota importante

No mencioné esto en la conferencia, pero es posible que debas iniciar sesión en el centro de HuggingFace si no lo has hecho antes.

1. Si aún no lo has hecho, crea una cuenta de HuggingFace **gratuita** en https://huggingface.co y navega a Configuración desde el menú de usuario en la parte superior derecha. Luego, crea un nuevo token de API y obtén permisos de escritura.

2. De regreso aquí en colab, presiona el ícono de "llave" en el panel lateral a la izquierda y agrega un nuevo secreto:
En el campo de nombre, coloca `HF_TOKEN`
En el campo de valor, coloca tu token actual: `hf_...`
Asegúrate de que el interruptor de acceso al cuaderno esté encendido.

3. Ejecuta la celda a continuación para iniciar sesión. Deberás hacer esto en cada uno de tus colabs. Es una forma realmente útil de administrar tus secretos sin necesidad de escribirlos en colab. También hay un atajo para sobrescribir la línea que aparece a continuación con:
`hf_token = "hf_...."`
Pero esta no es una buena práctica, ya que deberás tener cuidado de no compartir el colab. ¡Y una de las mejores cosas de los colabs es que puedes compartirlos!

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
# Sentiment Analysis

classifier = pipeline("sentiment-analysis", device="cuda")
result = classifier("¡Estoy súper emocionado de estar en camino hacia la maestría en LLM!")
print(result)

In [ ]:
# Named Entity Recognition

# `grouped_entities` se eliminó: el parámetro equivalente es `aggregation_strategy`
ner = pipeline("ner", aggregation_strategy="simple", device="cuda")
result = ner("Barack Obama fue el 44º presidente de los Estados Unidos.")
print(result)

In [ ]:
# Question Answering con Context
# La tarea "question-answering" ya no existe como pipeline: HuggingFace recomienda ahora usar un
# modelo de chat con el pipeline "text-generation". Creamos aquí ese pipeline y lo reutilizamos en las
# dos celdas siguientes (resumen y traducción) para no cargar el mismo modelo tres veces en la GPU.

chat_model = pipeline("text-generation", model="Qwen/Qwen3-4B-Instruct-2507", device="cuda")

messages = [{"role": "user", "content":
             "Contexto: Barack Obama fue el 44º presidente de los Estados Unidos.\n\n"
             "Pregunta: ¿Quién fue el 44º presidente de los Estados Unidos?"}]
result = chat_model(messages, max_new_tokens=50)
print(result[0]["generated_text"][-1]["content"])

In [ ]:
# Text Summarization

# La tarea "summarization" también desapareció: se resume pidiéndoselo al mismo modelo de chat.
text = """La biblioteca de transformadores Hugging Face es una herramienta increíblemente versátil y poderosa para el procesamiento del lenguaje natural (PLN).
Permite a los usuarios realizar una amplia gama de tareas, como la clasificación de texto, el reconocimiento de entidades con nombre y la respuesta a preguntas, entre otras.
Es una biblioteca extremadamente popular que se usa ampliamente en la comunidad de ciencia de datos de código abierto.
Reduce la barrera de entrada al campo al proporcionar a los científicos de datos una forma productiva y conveniente de trabajar con modelos de transformadores.
"""
messages = [{"role": "user", "content": f"Resume el siguiente texto en un par de frases:\n\n{text}"}]
summary = chat_model(messages, max_new_tokens=120)
print(summary[0]["generated_text"][-1]["content"])

In [ ]:
# Translation
# Las tareas "translation_en_to_XX" también se eliminaron: misma vía, el modelo de chat.

messages = [{"role": "user", "content":
             "Traduce al francés, responde solo con la traducción:\n\n"
             "Data Scientists were amazed by the power and simplicity of the API of pipelines from HuggingFace."}]
result = chat_model(messages, max_new_tokens=100)
print(result[0]["generated_text"][-1]["content"])

In [ ]:
# Classification

classifier = pipeline("zero-shot-classification", device="cuda")
result = classifier("¡La librería de Transformers de Hugging Face es increíble!",
                    candidate_labels=["tecnología", "deporte", "política"])
print(result)

In [ ]:
# Text Generation

# Fijamos gpt2 explícitamente: el modelo por defecto de `text-generation` ha cambiado a uno mucho más
# grande, y aquí queremos justamente ver lo pobre que es el resultado de un modelo pequeño.
generator = pipeline("text-generation", model="gpt2", device="cuda")
result = generator("Si hay algo que quiero que recuerdes sobre el uso de los pipelines de HuggingFace, es")
print(result[0]['generated_text'])

In [ ]:
# Image Generation

image_gen = DiffusionPipeline.from_pretrained(
    # stabilityai/stable-diffusion-2 fue retirado del Hub de HuggingFace (404): usamos SDXL
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16"
    ).to("cuda")

text = "Una clase de científicos de datos aprendiendo sobre IA, al estilo surrealista de Salvador Dalí"
image = image_gen(prompt=text).images[0]
image

In [ ]:
# Audio Generation

# El pipeline de una línea `pipeline("text-to-speech", ...)` está roto en la versión actual de
# transformers: su propio código llama a `.to(dtype=...)` sobre el resultado del procesador, y ese
# objeto ya no acepta `dtype` (TypeError: BatchEncoding.to() got an unexpected keyword argument
# 'dtype'). Es un fallo de la librería, no del modelo, así que usamos SpeechT5 directamente — que es
# exactamente lo que hace el pipeline por dentro: procesador -> modelo -> vocoder.

processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts").to("cuda")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan").to("cuda")

# `datasets` ya no ejecuta scripts de carga; este dataset tiene una revisión en Parquet que sí funciona
embeddings_dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation",
                                  revision="refs/convert/parquet")
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0).to("cuda")

inputs = processor(text="¡Hola a un ingeniero en inteligencia artificial, en camino hacia la maestría!",
                   return_tensors="pt").to("cuda")
speech = model.generate_speech(inputs["input_ids"], speaker_embedding, vocoder=vocoder)

sf.write("speech.wav", speech.cpu().numpy(), samplerate=16000)
Audio("speech.wav")


# Todos los pipelines disponibles

Aquí están todos los pipelines disponibles de Transformers y Diffusers.

¡Gracias al estudiante Lucky P por sugerirme que incluya esto!

Hay una lista de pipelines debajo de las Tareas en esta página (tienes que desplazarte un poco hacia abajo y luego expandir los parámetros para ver las Tareas):

https://huggingface.co/docs/transformers/main_classes/pipelines

También hay esta lista de Tareas para los modelos de Difusión en lugar de Transformadores, siguiendo el ejemplo de generación de imágenes donde uso DiffusionPipeline anterior.

https://huggingface.co/docs/diffusers/en/api/pipelines/overview

Si se te ocurren algunos ejemplos interesantes de otros pipelines, ¡compártelos conmigo! Es maravilloso cómo HuggingFace hace que esta funcionalidad avanzada de IA esté disponible para la inferencia con una API tan simple.